# Pairwise Fair Comparison for Multivariate Counterfactuals

This notebook reuses the multivariate evaluation setup and compares methods pairwise on the common instances they both evaluated. For non-validity metrics, only shared instances where both methods produced valid counterfactuals are used. NUN validity can be added as an optional extra filter, but it is not required by default.

In [1]:
cd ../..

D:\Users\mrefoyo\Proyectos\Sub-SpaCE_plus


D:\.conda_envs\mascots310\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from experiments.evaluation.evaluation_utils import (
    load_dataset_for_eval,
    calculate_metrics_for_dataset,
    summarize_target_pairwise_comparisons,
)

print(tf.__version__)

2.10.1


In [30]:
# DATASETS = ['CBF', 'chinatown', 'coffee', 'gunpoint', 'ECG200']
DATASETS = [
    "BasicMotions", 
    "NATOPS", "UWaveGestureLibrary",
    'ArticularyWordRecognition', 'Cricket', 'Epilepsy', 
    'PenDigits', 'PEMS-SF', 'RacketSports', 'SelfRegulationSCP1'
]

MO_UTILITY = np.array([0.1, 0.4 * 0.7, 0.6 * 0.7, 0.2])
MODEL_TO_EXPLAIN = 'inceptiontime_noscaling'
SCALING = 'none'
OSC_NAMES = {'AE': 'ae_basic_train', 'IF': 'if_basic_train', 'LOF': 'lof_basic_train'}
METHODS = {
    # 'comte_gpu': 'COMTE',
    # 'abcf_gpu': 'AB-CF',
    # 'discox_gpu': 'DiscoX',
    'mcels': 'M-CELS',
    # 'mascots_scalar_gpu': 'MASCOTS',
    'ec57e0a613241455487f3f40483cd34095c81bf7': 'Multi-SpaCE',
}

HIGHER_IS_BETTER_METRICS = {'valid'}
METRICS = ['valid', 'sparsity', 'L2', 'AE_OS', 'IF_OS', 'LOF_OS', 'subsequences', 'subsequences %', '(sparsity + subsequences %) / 2', 'times']
METRIC_NAME_MAP = {
    'valid': 'Validity',
    'sparsity': 'Sparsity',
    'L2': 'Proximity (L2)',
    '(sparsity + subsequences %) / 2': '(Sparsity + Subsequences [%]) / 2',
    'AE_OS': 'AE OS',
    'IF_OS': 'IF OS',
    'LOF_OS': 'LOF OS',
    'subsequences': 'Subsequences',
    'subsequences %': 'Subsequences %',
    'times': 'Time',
}

PENALIZATION_QUANTILES = [1.0]
PENALIZE_INVALID = False

## Get Results

This block is intentionally the same evaluation entry point used in the current multivariate notebook.

In [33]:
data_dict = {}
models_dict = {}
outlier_calculators_dict = {}
possible_nuns_dict = {}
desired_classes_dict = {}
original_classes_dict = {}

mean_results_dict_all = {}
methods_cfs_dict_all = {}
results_all_datasets_df_all = {}
common_test_indexes_dict_all = {}

for dataset in DATASETS:
    print(f'Calculating metrics for {dataset}')
    data_tuple, original_classes, model, outlier_calculators, possible_nuns, desired_classes = load_dataset_for_eval(
        dataset,
        MODEL_TO_EXPLAIN,
        OSC_NAMES,
        scaling=SCALING,
    )
    data_dict[dataset] = data_tuple
    models_dict[dataset] = model
    outlier_calculators_dict[dataset] = outlier_calculators
    possible_nuns_dict[dataset] = possible_nuns
    desired_classes_dict[dataset] = desired_classes
    original_classes_dict[dataset] = original_classes

    dataset_results = calculate_metrics_for_dataset(
        dataset,
        METHODS,
        MODEL_TO_EXPLAIN,
        data_tuple,
        original_classes,
        model,
        outlier_calculators,
        possible_nuns,
        mo_weights=MO_UTILITY,
        penalize_invalid=PENALIZE_INVALID,
        penalization_quantile=PENALIZATION_QUANTILES,
    )

    mean_results_dict_all[dataset] = {key: result['mean_std_df'] for key, result in dataset_results.items()}
    methods_cfs_dict_all[dataset] = {key: result['method_cfs_dataset'] for key, result in dataset_results.items()}
    results_all_datasets_df_all[dataset] = {key: result['results_df'] for key, result in dataset_results.items()}
    common_test_indexes_dict_all[dataset] = {key: result['common_test_indexes'] for key, result in dataset_results.items()}

Calculating metrics for BasicMotions
ec57e0a613241455487f3f40483cd34095c81bf7


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.93it/s]


mcels


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:04<00:00,  8.97it/s]


Calculating metrics for NATOPS
ec57e0a613241455487f3f40483cd34095c81bf7


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:14<00:00,  7.05it/s]


Calculating metrics for SelfRegulationSCP1
ec57e0a613241455487f3f40483cd34095c81bf7


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:29<00:00,  3.42it/s]


mcels


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:18<00:00,  5.40it/s]


In [34]:
results_all_datasets_df_all['BasicMotions']['none'].columns


Index(['ii', 'nchanges', 'sparsity', 'L1', 'L2', 'proba', 'valid',
       'nuns_valid', 'AE_OS', 'IF_OS', 'LOF_OS', 'AE_IOS', 'IF_IOS', 'LOF_IOS',
       'subsequences', 'subsequences %', '(sparsity + subsequences %) / 2',
       'times', 'method', 'best cf index', 'order', 'dataset'],
      dtype='object')

In [35]:
selected_penalization_key = 'none'

results_all_datasets_df = pd.concat(
    [results_all_datasets_df_all[dataset][selected_penalization_key] for dataset in DATASETS],
    ignore_index=True,
)

## Fair Pairwise Comparison

For each pair, comparisons are restricted to shared evaluated instances. For non-validity metrics, the compared instances must also be valid for both methods. NUN validity is not enforced unless explicitly requested.

In [37]:
TARGET_METHOD = 'Multi-SpaCE'

In [38]:
pairwise_summary_df, pairwise_detail_tables = summarize_target_pairwise_comparisons(
    results_df=results_all_datasets_df,
    target_method=TARGET_METHOD,
    metrics=METRICS,
    higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
    alpha=0.05,
    require_valid_nuns=False,
    require_both_valid_cfs=True,
)

pairwise_summary_df['metric_display'] = pairwise_summary_df['metric'].map(METRIC_NAME_MAP).fillna(pairwise_summary_df['metric'])
pairwise_summary_df['W/T/L'] = (
    pairwise_summary_df['win'].astype(str)
    + '/'
    + pairwise_summary_df['tie'].astype(str)
    + '/'
    + pairwise_summary_df['loss'].astype(str)
)
pairwise_summary_df

,metric,target_method,competitor_method,n_datasets,win,tie,loss,mean_shared_instances,min_shared_instances,max_shared_instances,...,wins_a,wins_b,ties,mean_delta_a_minus_b,median_delta_a_minus_b,better_method,p_value_holm,reject_holm,metric_display,W/T/L
0,(sparsity + subsequences %) / 2,Multi-SpaCE,M-CELS,10,1,0,9,60.6,7,97,...,1,9,0,-0.049270,-0.036982,M-CELS,0.009766,True,(Sparsity + Subsequences [%]) / 2,1/0/9
1,AE_OS,Multi-SpaCE,M-CELS,10,9,0,1,60.6,7,97,...,9,1,0,0.100373,0.112163,Multi-SpaCE,0.005859,True,AE OS,9/0/1
2,IF_OS,Multi-SpaCE,M-CELS,10,9,0,1,60.6,7,97,...,9,1,0,0.035001,0.030435,Multi-SpaCE,0.005859,True,IF OS,9/0/1
3,L2,Multi-SpaCE,M-CELS,10,2,0,8,60.6,7,97,...,2,8,0,-11.219525,-3.316836,M-CELS,0.083984,False,Proximity (L2),2/0/8
4,LOF_OS,Multi-SpaCE,M-CELS,10,9,0,1,60.6,7,97,...,9,1,0,0.048754,0.020647,Multi-SpaCE,0.037109,True,LOF OS,9/0/1
5,sparsity,Multi-SpaCE,M-CELS,10,0,0,10,60.6,7,97,...,0,10,0,-0.120580,-0.092666,M-CELS,0.001953,True,Sparsity,0/0/10
6,subsequences,Multi-SpaCE,M-CELS,10,7,0,3,60.6,7,97,...,7,3,0,-282.852083,1.387218,Multi-SpaCE,0.625000,False,Subsequences,7/0/3
7,subsequences %,Multi-SpaCE,M-CELS,10,7,0,3,60.6,7,97,...,7,3,0,0.022040,0.006603,Multi-SpaCE,0.232422,False,Subsequences %,7/0/3
8,times,Multi-SpaCE,M-CELS,10,8,0,2,60.6,7,97,...,8,2,0,-2.088086,13.361854,Multi-SpaCE,0.431641,False,Time,8/0/2
9,valid,Multi-SpaCE,M-CELS,10,10,0,0,91.2,40,100,...,10,0,0,0.329056,0.240278,Multi-SpaCE,0.001953,True,Validity,10/0/0


In [39]:
for metric in METRICS:
    metric_table = pairwise_summary_df.loc[
        pairwise_summary_df['metric'] == metric,
        [
            'competitor_method',
            'n_datasets',
            'W/T/L',
            'mean_shared_instances',
            'min_shared_instances',
            'max_shared_instances',
            'p_value_raw',
            'p_value_holm',
            'reject_holm',
            'better_method',
        ],
    ].copy()
    if metric_table.empty:
        continue
    metric_table = metric_table.sort_values('competitor_method').reset_index(drop=True)
    print(f"{METRIC_NAME_MAP.get(metric, metric)}: {TARGET_METHOD} vs rest")
    display(metric_table)

Validity: Multi-SpaCE vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS,10,10/0/0,91.2,40,100,0.001953,0.001953,True,Multi-SpaCE


Sparsity: Multi-SpaCE vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS,10,0/0/10,60.6,7,97,0.001953,0.001953,True,M-CELS


Proximity (L2): Multi-SpaCE vs rest


...

,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS,10,7/0/3,60.6,7,97,0.232422,0.232422,False,Multi-SpaCE


(Sparsity + Subsequences [%]) / 2: Multi-SpaCE vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS,10,1/0/9,60.6,7,97,0.009766,0.009766,True,M-CELS


Time: Multi-SpaCE vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS,10,8/0/2,60.6,7,97,0.431641,0.431641,False,Multi-SpaCE


In [40]:
for competitor_method, detail_df in pairwise_detail_tables.items():
    print(f'{TARGET_METHOD} vs {competitor_method}')
    detail_df = detail_df.copy()
    detail_df['metric_display'] = detail_df['metric'].map(METRIC_NAME_MAP).fillna(detail_df['metric'])
    display(
        detail_df[
            [
                'dataset',
                'metric_display',
                'n_common_instances',
                'n_common_valid_nuns',
                'n_metric_instances',
                'method_a_mean',
                'method_b_mean',
                'instance_wins_a',
                'instance_ties',
                'instance_wins_b',
                'dataset_winner',
            ]
        ].sort_values(['metric_display', 'dataset']).reset_index(drop=True)
    )

Multi-SpaCE vs M-CELS


,dataset,metric_display,n_common_instances,n_common_valid_nuns,n_metric_instances,method_a_mean,method_b_mean,instance_wins_a,instance_ties,instance_wins_b,dataset_winner
0,ArticularyWordRecognition,(Sparsity + Subsequences [%]) / 2,100,100,34,0.105177,0.027812,1,0,33,M-CELS
1,BasicMotions,(Sparsity + Subsequences [%]) / 2,40,40,28,0.119613,0.055506,3,0,25,M-CELS
2,Cricket,(Sparsity + Subsequences [%]) / 2,72,72,59,0.200751,0.046689,3,0,56,M-CELS
3,Epilepsy,(Sparsity + Subsequences [%]) / 2,100,100,39,0.147125,0.109514,15,1,23,M-CELS
4,NATOPS,(Sparsity + Subsequences [%]) / 2,100,100,97,0.063481,0.028692,24,3,70,M-CELS
...,...,...,...,...,...,...,...,...,...,...,...
95,PEMS-SF,Validity,100,100,100,1.000000,0.890000,11,89,0,Multi-SpaCE
96,PenDigits,Validity,100,100,100,1.000000,0.070000,93,7,0,Multi-SpaCE
97,RacketSports,Validity,100,100,100,1.000000,0.950000,5,95,0,Multi-SpaCE
98,SelfRegulationSCP1,Validity,100,100,100,1.000000,0.900000,10,90,0,Multi-SpaCE
